# Phase 4: Joint Two-Head Fine-Tuning

This notebook jointly fine-tunes a single shared MentalBERT backbone with two classification heads:

- **Head 1 (Condition)**: Normal, Depression, Anxiety, Stress, Suicidal (5 classes, trained on Mukherjee + Dreaddit)
- **Head 2 (Cause)**: No reason, Bias or abuse, Jobs and careers, Medication, Relationship, Alienation (6 classes, trained on CAMS)

**Why joint training?** Heads 1 and 2 were each trained on separate copies of MentalBERT. The final model needs one shared backbone serving both tasks. Joint training forces the backbone to learn representations useful for both condition detection and cause identification simultaneously.

**Strategy**: Load a fresh MentalBERT backbone, attach both classification heads (initialized from the individually trained checkpoints), then train on alternating batches from both datasets. A combined loss (weighted sum of Head 1 loss and Head 2 loss) updates the shared backbone and both heads together.

**Inputs**:
- Pre-split Head 1 CSVs (head1_train.csv, head1_val.csv, head1_test.csv)
- Pre-split CAMS CSVs (cams_train.csv, cams_val.csv, cams_test.csv)
- Individually trained Head 1 checkpoint (best_head1_checkpoint.pt)
- Individually trained Head 2 checkpoint (best_head2_checkpoint.pt)

**Outputs**:
- Joint model checkpoint with shared backbone + both heads
- Per-head test evaluation results
- Training curves comparing both heads across epochs

## Section 1: Environment Check

Check GPU availability and load the Hugging Face token from Kaggle Secrets. The joint model has the same memory footprint as a single head model (one backbone, two small linear heads) so it fits comfortably on a T4 or P100.

In [ ]:
import os
import sys
import json
import random
import warnings

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, Sampler
from torch.optim import AdamW
from transformers import AutoTokenizer, AutoModel, get_linear_schedule_with_warmup
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import (
    classification_report, confusion_matrix,
    f1_score, accuracy_score
)
from tqdm.auto import tqdm
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

warnings.filterwarnings("ignore", category=FutureWarning)

RANDOM_SEED = 42
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(RANDOM_SEED)

print("SECTION 1: ENVIRONMENT CHECK")

if torch.cuda.is_available():
    device = torch.device("cuda")
    gpu_name = torch.cuda.get_device_name(0)
    gpu_mem = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)
    print(f"GPU detected: {gpu_name}")
    print(f"VRAM: {gpu_mem:.1f} GB")
else:
    device = torch.device("cpu")
    print("WARNING: No GPU detected. Training on CPU is not practical.")
    print("Enable GPU: Settings > Accelerator > GPU T4 x2 or P100")

print(f"PyTorch version: {torch.__version__}")
print(f"Device: {device}")

# Load HF token from Kaggle Secrets
try:
    from kaggle_secrets import UserSecretsClient
    secrets = UserSecretsClient()
    HF_TOKEN = secrets.get_secret("HF_TOKEN")
    print("HF_TOKEN loaded from Kaggle Secrets.")
except Exception as e:
    HF_TOKEN = os.environ.get("HF_TOKEN")
    if HF_TOKEN:
        print("HF_TOKEN loaded from environment variable.")
    else:
        print(f"WARNING: Could not load HF_TOKEN: {e}")
        print("MentalBERT requires authentication. Add HF_TOKEN as a Kaggle Secret.")

CHECKPOINT_DIR = "/kaggle/working/checkpoints"
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
print(f"Checkpoint directory: {CHECKPOINT_DIR}")

## Section 2: Load and Inspect All Data

Load both datasets: Head 1 (Mukherjee + Dreaddit condition data) and Head 2 (CAMS cause data). Print row counts, class distributions, and verify columns are present. Both datasets must have a `cleaned_text` column for the shared tokenizer.

In [ ]:
print("SECTION 2: LOAD AND INSPECT ALL DATA")

# Update these paths to match your Kaggle dataset names
HEAD1_DIR = "/kaggle/input/mental-health-head1-combined"
CAMS_DIR = "/kaggle/input/mental-health-cams-combined"

# Head 1 data (Condition)
h1_train = pd.read_csv(os.path.join(HEAD1_DIR, "head1_train.csv"))
h1_val = pd.read_csv(os.path.join(HEAD1_DIR, "head1_val.csv"))
h1_test = pd.read_csv(os.path.join(HEAD1_DIR, "head1_test.csv"))

# CAMS data (Cause)
h2_train = pd.read_csv(os.path.join(CAMS_DIR, "cams_train.csv"))
h2_val = pd.read_csv(os.path.join(CAMS_DIR, "cams_val.csv"))
h2_test = pd.read_csv(os.path.join(CAMS_DIR, "cams_test.csv"))

print("\n--- HEAD 1 (Condition) ---")
for name, df in [("Train", h1_train), ("Val", h1_val), ("Test", h1_test)]:
    print(f"  {name}: {len(df):,d} rows")
    counts = df["condition_label"].value_counts()
    for label, count in counts.items():
        pct = 100 * count / len(df)
        print(f"    {label:20s}  {count:>7,d}  ({pct:5.1f}%)")

print("\n--- HEAD 2 (Cause / CAMS) ---")
for name, df in [("Train", h2_train), ("Val", h2_val), ("Test", h2_test)]:
    print(f"  {name}: {len(df):,d} rows")
    counts = df["category_name"].value_counts()
    for label, count in counts.items():
        pct = 100 * count / len(df)
        print(f"    {label:20s}  {count:>7,d}  ({pct:5.1f}%)")

print("\nSize ratio Head1:Head2 =", f"{len(h1_train)/len(h2_train):.1f}:1")
print("Joint training will need to balance batch sampling across these two datasets.")

## Section 3: Label Encoding

Set up label mappings for both heads. These must exactly match the mappings used during individual head training so that the classifier layer weights load correctly.

In [ ]:
print("SECTION 3: LABEL ENCODING")

# Head 1: Condition labels (must match Phase 1 training)
H1_LABEL_TO_ID = {
    "Normal": 0,
    "Depression": 1,
    "Anxiety": 2,
    "Stress": 3,
    "Suicidal": 4,
}
H1_ID_TO_LABEL = {v: k for k, v in H1_LABEL_TO_ID.items()}
H1_NUM_CLASSES = len(H1_LABEL_TO_ID)
H1_LABEL_NAMES = [H1_ID_TO_LABEL[i] for i in range(H1_NUM_CLASSES)]

# Head 2: Cause labels (must match Phase 2 training)
H2_LABEL_TO_ID = {
    "No reason": 0,
    "Bias or abuse": 1,
    "Jobs and careers": 2,
    "Medication": 3,
    "Relationship": 4,
    "Alienation": 5,
}
H2_ID_TO_LABEL = {v: k for k, v in H2_LABEL_TO_ID.items()}
H2_NUM_CLASSES = len(H2_LABEL_TO_ID)
H2_LABEL_NAMES = [H2_ID_TO_LABEL[i] for i in range(H2_NUM_CLASSES)]

print("\nHead 1 (Condition) label mapping:")
for label, idx in H1_LABEL_TO_ID.items():
    print(f"  {label} -> {idx}")

print("\nHead 2 (Cause) label mapping:")
for label, idx in H2_LABEL_TO_ID.items():
    print(f"  {label} -> {idx}")

# Apply encoding
for df in [h1_train, h1_val, h1_test]:
    df["label"] = df["condition_label"].map(H1_LABEL_TO_ID)
for df in [h2_train, h2_val, h2_test]:
    df["label"] = df["category_name"].map(H2_LABEL_TO_ID)

# Verify no unmapped labels
for name, df in [("H1 Train", h1_train), ("H1 Val", h1_val), ("H1 Test", h1_test),
                  ("H2 Train", h2_train), ("H2 Val", h2_val), ("H2 Test", h2_test)]:
    n_null = df["label"].isna().sum()
    status = "OK" if n_null == 0 else f"WARNING: {n_null} unmapped"
    print(f"  {name:12s}: {status}")

# Save combined label mappings
joint_mapping = {
    "head1_condition": {str(k): v for k, v in H1_ID_TO_LABEL.items()},
    "head2_cause": {str(k): v for k, v in H2_ID_TO_LABEL.items()},
}
mapping_path = os.path.join(CHECKPOINT_DIR, "joint_label_mapping.json")
with open(mapping_path, "w") as f:
    json.dump(joint_mapping, f, indent=2)
print(f"\nJoint label mapping saved to {mapping_path}")

## Section 4: Alternating Batch Sampler

The Head 1 dataset is ~10x larger than CAMS. If we just merged them, the model would see Head 1 data far more often and the shared backbone would be biased toward condition classification.

Instead, training alternates: one batch of Head 1 data, then one batch of Head 2 data, repeat. The CAMS data cycles multiple times per epoch to match. This ensures the backbone receives roughly equal gradient signal from both tasks.

Head 1 also uses domain-balanced sampling (equal Mukherjee and Dreaddit per batch) as in Phase 1.

In [ ]:
print("SECTION 4: ALTERNATING BATCH SAMPLER")


class DomainBalancedSampler(Sampler):
    """
    Sampler for Head 1 that balances Mukherjee and Dreaddit sources
    within each batch. Same logic as used in Phase 1 training.
    
    Each batch contains half Mukherjee indices and half Dreaddit indices.
    The larger source determines epoch length; the smaller source repeats.
    """

    def __init__(self, source_labels, batch_size):
        super().__init__()
        self.batch_size = batch_size
        self.half_batch = batch_size // 2

        self.muk_indices = [
            i for i, s in enumerate(source_labels) if s == "mukherjee"
        ]
        self.dread_indices = [
            i for i, s in enumerate(source_labels) if s == "dreaddit"
        ]

        n_muk_batches = len(self.muk_indices) // self.half_batch
        n_dread_batches = len(self.dread_indices) // self.half_batch
        self.n_batches = max(n_muk_batches, n_dread_batches)

        muk_samples = self.n_batches * self.half_batch
        dread_samples = self.n_batches * self.half_batch
        print(f"  DomainBalancedSampler (Head 1):")
        print(f"    Mukherjee:  {len(self.muk_indices):,d} indices, "
              f"sampled {muk_samples:,d}/epoch ({muk_samples/len(self.muk_indices):.1f}x)")
        print(f"    Dreaddit:   {len(self.dread_indices):,d} indices, "
              f"sampled {dread_samples:,d}/epoch ({dread_samples/len(self.dread_indices):.1f}x)")
        print(f"    Batches/epoch: {self.n_batches:,d}")

    def _extend_and_shuffle(self, indices, target_length):
        pool = indices.copy()
        random.shuffle(pool)
        repeats = (target_length // len(pool)) + 1
        return (pool * repeats)[:target_length]

    def __iter__(self):
        target = self.n_batches * self.half_batch
        muk_pool = self._extend_and_shuffle(self.muk_indices, target)
        dread_pool = self._extend_and_shuffle(self.dread_indices, target)
        all_indices = []
        for i in range(self.n_batches):
            s = i * self.half_batch
            e = s + self.half_batch
            batch = muk_pool[s:e] + dread_pool[s:e]
            random.shuffle(batch)
            all_indices.extend(batch)
        return iter(all_indices)

    def __len__(self):
        return self.n_batches * self.batch_size


# Head 1 uses domain balanced sampler
h1_source_labels = h1_train["source_dataset"].tolist()
print()

# Head 2 uses simple shuffle (single source)
print(f"  Head 2 (CAMS): {len(h2_train):,d} training samples, standard shuffle")
h1_batches_per_epoch = max(
    len([s for s in h1_source_labels if s == "mukherjee"]) // 8,
    len([s for s in h1_source_labels if s == "dreaddit"]) // 8,
)
h2_batches_per_epoch = len(h2_train) // 16
cams_repeats = h1_batches_per_epoch / max(h2_batches_per_epoch, 1)
print(f"\n  Per epoch: ~{h1_batches_per_epoch:,d} Head 1 batches, ~{h2_batches_per_epoch:,d} Head 2 batches")
print(f"  CAMS will cycle ~{cams_repeats:.1f}x per epoch to match Head 1 batch count")

## Section 5: Dataset and DataLoader

Both heads share the same tokenizer and max token length (256). Each dataset returns its own integer labels. During training, the alternating loop draws one batch from Head 1, then one from Head 2, computing the appropriate loss for each.

In [ ]:
print("SECTION 5: DATASET AND DATALOADER")

MAX_LENGTH = 256
TRAIN_BATCH_SIZE = 16
EVAL_BATCH_SIZE = 32


class TextDataset(Dataset):
    """Tokenizes cleaned text and returns input_ids, attention_mask, and integer label."""

    def __init__(self, dataframe, tokenizer, max_length=256):
        self.texts = dataframe["cleaned_text"].tolist()
        self.labels = dataframe["label"].tolist()
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = str(self.texts[idx]) if self.texts[idx] is not None else ""
        encoding = self.tokenizer(
            text,
            max_length=self.max_length,
            padding="max_length",
            truncation=True,
            return_tensors="pt",
        )
        return {
            "input_ids": encoding["input_ids"].squeeze(0),
            "attention_mask": encoding["attention_mask"].squeeze(0),
            "label": torch.tensor(self.labels[idx], dtype=torch.long),
        }


# Load shared tokenizer
print("Loading MentalBERT tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(
    "mental/mental-bert-base-uncased", token=HF_TOKEN
)
print(f"Tokenizer loaded. Vocab size: {tokenizer.vocab_size}")

# Create datasets
h1_train_ds = TextDataset(h1_train, tokenizer, MAX_LENGTH)
h1_val_ds = TextDataset(h1_val, tokenizer, MAX_LENGTH)
h1_test_ds = TextDataset(h1_test, tokenizer, MAX_LENGTH)

h2_train_ds = TextDataset(h2_train, tokenizer, MAX_LENGTH)
h2_val_ds = TextDataset(h2_val, tokenizer, MAX_LENGTH)
h2_test_ds = TextDataset(h2_test, tokenizer, MAX_LENGTH)

print(f"Head 1 datasets: train={len(h1_train_ds):,d}, val={len(h1_val_ds):,d}, test={len(h1_test_ds):,d}")
print(f"Head 2 datasets: train={len(h2_train_ds):,d}, val={len(h2_val_ds):,d}, test={len(h2_test_ds):,d}")

# Training loaders
h1_train_sampler = DomainBalancedSampler(h1_source_labels, TRAIN_BATCH_SIZE)
h1_train_loader = DataLoader(
    h1_train_ds, batch_size=TRAIN_BATCH_SIZE,
    sampler=h1_train_sampler, num_workers=2, pin_memory=True
)
h2_train_loader = DataLoader(
    h2_train_ds, batch_size=TRAIN_BATCH_SIZE,
    shuffle=True, num_workers=2, pin_memory=True, drop_last=True
)

# Val and test loaders (sequential, no balancing needed)
h1_val_loader = DataLoader(h1_val_ds, batch_size=EVAL_BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
h1_test_loader = DataLoader(h1_test_ds, batch_size=EVAL_BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
h2_val_loader = DataLoader(h2_val_ds, batch_size=EVAL_BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
h2_test_loader = DataLoader(h2_test_ds, batch_size=EVAL_BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

print(f"\nHead 1 train loader: {len(h1_train_loader):,d} batches")
print(f"Head 2 train loader: {len(h2_train_loader):,d} batches")

## Section 6: Joint Two-Head Model

The model has one shared MentalBERT backbone and two separate classification heads. Each head is a Dropout + Linear layer, identical in structure to the individually trained heads.

The backbone starts fresh from MentalBERT. The two classifier heads are initialized from the individually trained checkpoints (Phase 1 and Phase 2). This gives the heads a warm start so the backbone only needs to learn how to serve both tasks, not retrain the heads from scratch.

In [ ]:
print("SECTION 6: JOINT TWO-HEAD MODEL")


class JointTwoHeadClassifier(nn.Module):
    """
    Shared MentalBERT backbone with two classification heads.
    Head 1 classifies mental health conditions (5 classes).
    Head 2 classifies causes (6 classes).
    Both heads use the CLS token pooled output from the shared backbone.
    """

    def __init__(self, backbone, head1_classes=5, head2_classes=6, dropout_rate=0.1):
        super().__init__()
        self.backbone = backbone
        self.dropout = nn.Dropout(dropout_rate)

        # Head 1: Condition classifier
        self.head1_classifier = nn.Linear(768, head1_classes)

        # Head 2: Cause classifier
        self.head2_classifier = nn.Linear(768, head2_classes)

    def forward(self, input_ids, attention_mask, head="head1"):
        """
        Forward pass through the shared backbone, then route to
        the specified head. Returns logits for that head only.
        """
        outputs = self.backbone(input_ids=input_ids, attention_mask=attention_mask)
        pooled = self.dropout(outputs.pooler_output)

        if head == "head1":
            return self.head1_classifier(pooled)
        elif head == "head2":
            return self.head2_classifier(pooled)
        else:
            raise ValueError(f"Unknown head: {head}. Use 'head1' or 'head2'.")


# Load fresh MentalBERT backbone
print("Loading fresh MentalBERT backbone...")
backbone = AutoModel.from_pretrained(
    "mental/mental-bert-base-uncased", token=HF_TOKEN
)
print("MentalBERT loaded.")

# Build the joint model
model = JointTwoHeadClassifier(
    backbone, head1_classes=H1_NUM_CLASSES,
    head2_classes=H2_NUM_CLASSES, dropout_rate=0.1
)
model = model.to(device)

total_params = sum(p.numel() for p in model.parameters())
backbone_params = sum(p.numel() for p in model.backbone.parameters())
head1_params = sum(p.numel() for p in model.head1_classifier.parameters())
head2_params = sum(p.numel() for p in model.head2_classifier.parameters())
print(f"\nParameter breakdown:")
print(f"  Backbone (shared): {backbone_params:,d}")
print(f"  Head 1 classifier: {head1_params:,d}")
print(f"  Head 2 classifier: {head2_params:,d}")
print(f"  Total:             {total_params:,d}")

## Section 7: Initialize Heads from Individual Checkpoints

Load the individually trained Head 1 and Head 2 checkpoints. Extract only the classifier layer weights (not the backbone weights) and use them to initialize the corresponding heads in the joint model.

This gives the heads a warm start. The backbone starts fresh and must learn to serve both tasks during joint training.

**Important**: The checkpoint paths below must point to where you uploaded the .pt files on Kaggle. Upload them as a Kaggle dataset.

In [ ]:
print("SECTION 7: INITIALIZE HEADS FROM CHECKPOINTS")

# Paths to the individually trained checkpoints
# Upload both .pt files as a Kaggle dataset and update these paths
HEAD1_CKPT_PATH = "/kaggle/input/mental-health-checkpoints/best_head1_checkpoint.pt"
HEAD2_CKPT_PATH = "/kaggle/input/mental-health-checkpoints/best_head2_checkpoint.pt"

# Load Head 1 checkpoint and transfer classifier weights
try:
    h1_ckpt = torch.load(HEAD1_CKPT_PATH, map_location=device, weights_only=False)
    h1_state = h1_ckpt["model_state_dict"]

    # The individually trained model used "classifier.weight" and "classifier.bias"
    # Map these to our joint model's "head1_classifier.weight" and "head1_classifier.bias"
    model.head1_classifier.weight.data.copy_(h1_state["classifier.weight"])
    model.head1_classifier.bias.data.copy_(h1_state["classifier.bias"])
    print(f"Head 1 classifier initialized from checkpoint (epoch {h1_ckpt.get('epoch', '?')})")
    print(f"  Original val macro F1: {h1_ckpt.get('val_macro_f1', 'N/A')}")
except FileNotFoundError:
    print("WARNING: Head 1 checkpoint not found. Head 1 classifier will train from scratch.")
except KeyError as e:
    print(f"WARNING: Could not find expected keys in Head 1 checkpoint: {e}")
    print("  Trying alternative key names...")
    try:
        # If the key names differ, try common alternatives
        h1_keys = [k for k in h1_state.keys() if "classifier" in k]
        print(f"  Available classifier keys: {h1_keys}")
    except:
        pass

# Load Head 2 checkpoint and transfer classifier weights
try:
    h2_ckpt = torch.load(HEAD2_CKPT_PATH, map_location=device, weights_only=False)
    h2_state = h2_ckpt["model_state_dict"]

    model.head2_classifier.weight.data.copy_(h2_state["classifier.weight"])
    model.head2_classifier.bias.data.copy_(h2_state["classifier.bias"])
    print(f"\nHead 2 classifier initialized from checkpoint (epoch {h2_ckpt.get('epoch', '?')})")
    print(f"  Original val macro F1: {h2_ckpt.get('val_macro_f1', 'N/A')}")
except FileNotFoundError:
    print("WARNING: Head 2 checkpoint not found. Head 2 classifier will train from scratch.")
except KeyError as e:
    print(f"WARNING: Could not find expected keys in Head 2 checkpoint: {e}")
    try:
        h2_keys = [k for k in h2_state.keys() if "classifier" in k]
        print(f"  Available classifier keys: {h2_keys}")
    except:
        pass

print("\nBackbone starts fresh from MentalBERT (not from either individual checkpoint).")
print("Joint training will adapt the backbone to serve both tasks simultaneously.")

## Section 8: Loss Functions

Each head gets its own weighted cross-entropy loss, computed from its respective training set class distribution. The total loss per training step is a weighted combination of whichever head's batch is being processed.

Since the datasets are very different in size (Head 1 ~42k rows, Head 2 ~4k rows), we scale the Head 2 loss slightly higher so it has proportional influence on the shared backbone.

In [ ]:
print("SECTION 8: LOSS FUNCTIONS")

# Head 1 class weights
h1_labels = h1_train["label"].values
h1_class_weights = compute_class_weight(
    class_weight="balanced",
    classes=np.array(sorted(H1_LABEL_TO_ID.values())),
    y=h1_labels,
)
print("\nHead 1 (Condition) class weights:")
for idx, w in enumerate(h1_class_weights):
    print(f"  {H1_ID_TO_LABEL[idx]:15s}: {w:.4f}")

h1_weights_tensor = torch.tensor(h1_class_weights, dtype=torch.float32).to(device)
h1_criterion = nn.CrossEntropyLoss(weight=h1_weights_tensor)

# Head 2 class weights
h2_labels = h2_train["label"].values
h2_class_weights = compute_class_weight(
    class_weight="balanced",
    classes=np.array(sorted(H2_LABEL_TO_ID.values())),
    y=h2_labels,
)
print("\nHead 2 (Cause) class weights:")
for idx, w in enumerate(h2_class_weights):
    print(f"  {H2_ID_TO_LABEL[idx]:20s}: {w:.4f}")

h2_weights_tensor = torch.tensor(h2_class_weights, dtype=torch.float32).to(device)
h2_criterion = nn.CrossEntropyLoss(weight=h2_weights_tensor)

# Loss combination weight: scale Head 2 higher since its dataset is ~10x smaller
# This gives the smaller dataset proportionally more influence on the backbone
HEAD2_LOSS_SCALE = 2.0
print(f"\nHead 2 loss scaling factor: {HEAD2_LOSS_SCALE}")
print("This amplifies Head 2 gradients so the smaller CAMS dataset has")
print("meaningful influence on the shared backbone during joint training.")

## Section 8.5: Training Tracker Initialization

Initialize history dictionaries that record metrics across epochs for both heads. These are used to generate training curves at the end.

In [ ]:
history = {
    "epoch": [],
    "train_loss_total": [],
    "train_loss_h1": [],
    "train_loss_h2": [],
    # Head 1 val metrics
    "h1_val_macro_f1": [],
    "h1_val_accuracy": [],
    "h1_val_normal_f1": [],
    "h1_val_depression_f1": [],
    "h1_val_anxiety_f1": [],
    "h1_val_stress_f1": [],
    "h1_val_suicidal_f1": [],
    # Head 2 val metrics
    "h2_val_macro_f1": [],
    "h2_val_accuracy": [],
    "h2_val_no_reason_f1": [],
    "h2_val_bias_or_abuse_f1": [],
    "h2_val_jobs_and_careers_f1": [],
    "h2_val_medication_f1": [],
    "h2_val_relationship_f1": [],
    "h2_val_alienation_f1": [],
}
print("Training history tracker initialized for both heads.")

## Section 9: Training Loop

The joint training loop alternates between Head 1 and Head 2 batches:

1. Draw a batch from Head 1 data, compute Head 1 loss, backprop
2. Draw a batch from Head 2 data, compute scaled Head 2 loss, backprop
3. Step the optimizer and scheduler after both

Head 2's data loader cycles when exhausted (since it is much smaller). Validation runs on both heads after every epoch. The best checkpoint is saved based on the **average** of both heads' macro F1, so neither head is neglected.

Early stopping triggers if the combined metric does not improve for 3 consecutive epochs.

In [ ]:
def evaluate_head(model, dataloader, device, head_name):
    """Run inference for a specific head and return predictions and true labels."""
    model.eval()
    all_preds = []
    all_labels = []
    with torch.no_grad():
        for batch in tqdm(dataloader, desc=f"Eval {head_name}", leave=False):
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["label"].to(device)
            logits = model(input_ids, attention_mask, head=head_name)
            preds = torch.argmax(logits, dim=1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
    return np.array(all_preds), np.array(all_labels)


def compute_metrics(preds, labels, label_names):
    """Compute classification metrics for a single head."""
    macro_f1 = f1_score(labels, preds, average="macro")
    accuracy = accuracy_score(labels, preds)
    report_dict = classification_report(
        labels, preds, target_names=label_names,
        output_dict=True, zero_division=0
    )
    report_str = classification_report(
        labels, preds, target_names=label_names, zero_division=0
    )
    cm = confusion_matrix(labels, preds)
    return {
        "macro_f1": macro_f1,
        "accuracy": accuracy,
        "report_dict": report_dict,
        "report_str": report_str,
        "confusion_matrix": cm,
    }

In [ ]:
print("SECTION 9: TRAINING LOOP")

# Hyperparameters
LEARNING_RATE = 1e-5
MAX_EPOCHS = 10
WARMUP_PROPORTION = 0.1
PATIENCE = 3

# The number of training steps per epoch is determined by Head 1 (the larger dataset)
steps_per_epoch = len(h1_train_loader)
total_steps = steps_per_epoch * MAX_EPOCHS
warmup_steps = int(total_steps * WARMUP_PROPORTION)

optimizer = AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=0.01)
scheduler = get_linear_schedule_with_warmup(
    optimizer, num_warmup_steps=warmup_steps, num_training_steps=total_steps
)

print(f"Optimizer: AdamW (lr={LEARNING_RATE}, weight_decay=0.01)")
print(f"Steps per epoch: {steps_per_epoch:,d} (set by Head 1 loader)")
print(f"Total training steps: {total_steps:,d}")
print(f"Warmup steps: {warmup_steps:,d}")
print(f"Max epochs: {MAX_EPOCHS}, Early stopping patience: {PATIENCE}")
print(f"Head 2 loss scale: {HEAD2_LOSS_SCALE}")

best_combined_f1 = 0.0
patience_counter = 0
best_epoch = 0
checkpoint_path = os.path.join(CHECKPOINT_DIR, "best_joint_checkpoint.pt")

for epoch in range(1, MAX_EPOCHS + 1):
    print(f"\n{'=' * 60}")
    print(f"EPOCH {epoch}/{MAX_EPOCHS}")
    print(f"{'=' * 60}")

    model.train()
    total_loss = 0.0
    h1_epoch_loss = 0.0
    h2_epoch_loss = 0.0
    n_steps = 0

    # Create a cycling iterator for Head 2 (since it is smaller)
    h2_iter = iter(h2_train_loader)

    progress = tqdm(h1_train_loader, desc=f"Joint training epoch {epoch}", leave=True)
    for h1_batch in progress:
        # Get Head 2 batch (cycle if exhausted)
        try:
            h2_batch = next(h2_iter)
        except StopIteration:
            h2_iter = iter(h2_train_loader)
            h2_batch = next(h2_iter)

        optimizer.zero_grad()

        # Forward + loss for Head 1
        h1_ids = h1_batch["input_ids"].to(device)
        h1_mask = h1_batch["attention_mask"].to(device)
        h1_labels_batch = h1_batch["label"].to(device)
        h1_logits = model(h1_ids, h1_mask, head="head1")
        loss_h1 = h1_criterion(h1_logits, h1_labels_batch)

        # Forward + loss for Head 2 (scaled)
        h2_ids = h2_batch["input_ids"].to(device)
        h2_mask = h2_batch["attention_mask"].to(device)
        h2_labels_batch = h2_batch["label"].to(device)
        h2_logits = model(h2_ids, h2_mask, head="head2")
        loss_h2 = h2_criterion(h2_logits, h2_labels_batch) * HEAD2_LOSS_SCALE

        # Combined loss
        loss = loss_h1 + loss_h2
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        scheduler.step()

        total_loss += loss.item()
        h1_epoch_loss += loss_h1.item()
        h2_epoch_loss += loss_h2.item()
        n_steps += 1
        progress.set_postfix({
            "h1_loss": f"{loss_h1.item():.3f}",
            "h2_loss": f"{loss_h2.item():.3f}"
        })

    avg_total_loss = total_loss / n_steps
    avg_h1_loss = h1_epoch_loss / n_steps
    avg_h2_loss = h2_epoch_loss / n_steps

    # Validate Head 1
    h1_preds, h1_true = evaluate_head(model, h1_val_loader, device, "head1")
    h1_metrics = compute_metrics(h1_preds, h1_true, H1_LABEL_NAMES)

    # Validate Head 2
    h2_preds, h2_true = evaluate_head(model, h2_val_loader, device, "head2")
    h2_metrics = compute_metrics(h2_preds, h2_true, H2_LABEL_NAMES)

    combined_f1 = (h1_metrics["macro_f1"] + h2_metrics["macro_f1"]) / 2

    # Print epoch summary
    print(f"\n  Epoch {epoch} Summary:")
    print(f"    Total Loss: {avg_total_loss:.4f}  (H1: {avg_h1_loss:.4f}, H2: {avg_h2_loss:.4f})")
    print(f"\n    HEAD 1 (Condition) Val Macro F1: {h1_metrics['macro_f1']:.4f}  Acc: {h1_metrics['accuracy']:.4f}")
    print(f"    {'Class':15s} {'F1':>8s} {'Prec':>8s} {'Rec':>8s}")
    print(f"    {'-' * 41}")
    for name in H1_LABEL_NAMES:
        cls = h1_metrics["report_dict"][name]
        print(f"    {name:15s} {cls['f1-score']:8.4f} {cls['precision']:8.4f} {cls['recall']:8.4f}")

    print(f"\n    HEAD 2 (Cause) Val Macro F1: {h2_metrics['macro_f1']:.4f}  Acc: {h2_metrics['accuracy']:.4f}")
    print(f"    {'Class':20s} {'F1':>8s} {'Prec':>8s} {'Rec':>8s}")
    print(f"    {'-' * 46}")
    for name in H2_LABEL_NAMES:
        cls = h2_metrics["report_dict"][name]
        print(f"    {name:20s} {cls['f1-score']:8.4f} {cls['precision']:8.4f} {cls['recall']:8.4f}")

    print(f"\n    Combined Macro F1 (avg of both heads): {combined_f1:.4f}")

    # Stress warning (Head 1)
    stress_f1 = h1_metrics["report_dict"]["Stress"]["f1-score"]
    if stress_f1 < 0.40:
        print(f"    WARNING: Head 1 Stress F1 ({stress_f1:.4f}) below 0.40")

    # Record history
    history["epoch"].append(epoch)
    history["train_loss_total"].append(avg_total_loss)
    history["train_loss_h1"].append(avg_h1_loss)
    history["train_loss_h2"].append(avg_h2_loss)
    history["h1_val_macro_f1"].append(h1_metrics["macro_f1"])
    history["h1_val_accuracy"].append(h1_metrics["accuracy"])
    history["h2_val_macro_f1"].append(h2_metrics["macro_f1"])
    history["h2_val_accuracy"].append(h2_metrics["accuracy"])
    for cls_name in H1_LABEL_NAMES:
        key = f"h1_val_{cls_name.lower()}_f1"
        history[key].append(h1_metrics["report_dict"][cls_name]["f1-score"])
    for cls_name in H2_LABEL_NAMES:
        key = f"h2_val_{cls_name.lower().replace(' ', '_')}_f1"
        history[key].append(h2_metrics["report_dict"][cls_name]["f1-score"])

    # Save best checkpoint
    if combined_f1 > best_combined_f1:
        best_combined_f1 = combined_f1
        best_epoch = epoch
        patience_counter = 0
        torch.save({
            "epoch": epoch,
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "combined_macro_f1": best_combined_f1,
            "h1_val_macro_f1": h1_metrics["macro_f1"],
            "h2_val_macro_f1": h2_metrics["macro_f1"],
            "h1_label_mapping": H1_ID_TO_LABEL,
            "h2_label_mapping": H2_ID_TO_LABEL,
        }, checkpoint_path)
        print(f"\n    NEW BEST checkpoint saved (combined F1: {best_combined_f1:.4f})")
    else:
        patience_counter += 1
        print(f"\n    No improvement. Best: {best_combined_f1:.4f} at epoch {best_epoch}. "
              f"Patience: {patience_counter}/{PATIENCE}")

    if patience_counter >= PATIENCE:
        print(f"\n  EARLY STOPPING triggered after {PATIENCE} epochs without improvement.")
        print(f"  Best combined macro F1: {best_combined_f1:.4f} at epoch {best_epoch}")
        break

print(f"\n{'=' * 60}")
print(f"TRAINING COMPLETE")
print(f"Best epoch: {best_epoch}")
print(f"Best combined macro F1: {best_combined_f1:.4f}")
print(f"Checkpoint: {checkpoint_path}")
print(f"{'=' * 60}")

## Section 10: Test Set Evaluation (Both Heads)

Load the best joint checkpoint and evaluate both heads on their respective held-out test sets. Compare results against the individual Phase 1 and Phase 2 baselines to check whether joint training helped, hurt, or maintained each head's performance.

In [ ]:
print("SECTION 10: TEST SET EVALUATION")

# Load best checkpoint
print(f"Loading best joint checkpoint from epoch {best_epoch}...")
ckpt = torch.load(checkpoint_path, map_location=device, weights_only=False)
model.load_state_dict(ckpt["model_state_dict"])
print(f"Loaded. Combined F1 was: {ckpt['combined_macro_f1']:.4f}")

# Evaluate Head 1
h1_test_preds, h1_test_true = evaluate_head(model, h1_test_loader, device, "head1")
h1_test_metrics = compute_metrics(h1_test_preds, h1_test_true, H1_LABEL_NAMES)

# Evaluate Head 2
h2_test_preds, h2_test_true = evaluate_head(model, h2_test_loader, device, "head2")
h2_test_metrics = compute_metrics(h2_test_preds, h2_test_true, H2_LABEL_NAMES)

# Print Head 1 results
print(f"\n{'='*50}")
print(f"HEAD 1 (CONDITION) TEST RESULTS")
print(f"{'='*50}")
print(f"Macro F1:  {h1_test_metrics['macro_f1']:.4f}")
print(f"Accuracy:  {h1_test_metrics['accuracy']:.4f}")
print(f"\nClassification Report:")
print(h1_test_metrics["report_str"])
print(f"Confusion Matrix (label order: {H1_LABEL_NAMES}):")
print(h1_test_metrics["confusion_matrix"])

# Print Head 2 results
print(f"\n{'='*50}")
print(f"HEAD 2 (CAUSE) TEST RESULTS")
print(f"{'='*50}")
print(f"Macro F1:  {h2_test_metrics['macro_f1']:.4f}")
print(f"Accuracy:  {h2_test_metrics['accuracy']:.4f}")
print(f"\nClassification Report:")
print(h2_test_metrics["report_str"])
print(f"Confusion Matrix (label order: {H2_LABEL_NAMES}):")
print(h2_test_metrics["confusion_matrix"])

# Compare to individual baselines
print(f"\n{'='*50}")
print(f"COMPARISON TO INDIVIDUAL TRAINING")
print(f"{'='*50}")
print(f"{'Metric':30s} {'Phase 1/2':>10s} {'Joint':>10s} {'Delta':>10s}")
print(f"{'-'*62}")

h1_baseline = 0.8272
h2_baseline = 0.5091
h1_delta = h1_test_metrics["macro_f1"] - h1_baseline
h2_delta = h2_test_metrics["macro_f1"] - h2_baseline

print(f"{'Head 1 (Condition) Macro F1':30s} {h1_baseline:10.4f} {h1_test_metrics['macro_f1']:10.4f} {h1_delta:+10.4f}")
print(f"{'Head 2 (Cause) Macro F1':30s} {h2_baseline:10.4f} {h2_test_metrics['macro_f1']:10.4f} {h2_delta:+10.4f}")

if h1_delta < -0.05:
    print(f"\nWARNING: Head 1 dropped more than 5 points. Joint training may be hurting condition classification.")
    print("Consider reducing HEAD2_LOSS_SCALE or freezing the backbone for initial epochs.")
if h2_delta < -0.05:
    print(f"\nWARNING: Head 2 dropped more than 5 points. Joint training may be hurting cause classification.")
    print("Consider increasing HEAD2_LOSS_SCALE.")
if h1_delta >= -0.02 and h2_delta >= -0.02:
    print(f"\nBoth heads maintained or improved performance during joint training.")

# Save results
results_path = os.path.join(CHECKPOINT_DIR, "joint_test_results.txt")
with open(results_path, "w") as f:
    f.write("PHASE 4: JOINT TWO-HEAD TEST RESULTS\n")
    f.write("=" * 50 + "\n\n")
    f.write(f"Best epoch: {best_epoch}\n")
    f.write(f"Combined macro F1: {ckpt['combined_macro_f1']:.4f}\n\n")

    f.write("HEAD 1 (CONDITION CLASSIFIER)\n")
    f.write(f"Macro F1: {h1_test_metrics['macro_f1']:.4f}\n")
    f.write(f"Accuracy: {h1_test_metrics['accuracy']:.4f}\n")
    f.write(f"Baseline (Phase 1): {h1_baseline:.4f}  Delta: {h1_delta:+.4f}\n\n")
    f.write(h1_test_metrics["report_str"])
    f.write(f"\nConfusion Matrix ({H1_LABEL_NAMES}):\n")
    for row in h1_test_metrics["confusion_matrix"]:
        f.write("  " + "  ".join(f"{v:>5d}" for v in row) + "\n")

    f.write(f"\n\nHEAD 2 (CAUSE CLASSIFIER)\n")
    f.write(f"Macro F1: {h2_test_metrics['macro_f1']:.4f}\n")
    f.write(f"Accuracy: {h2_test_metrics['accuracy']:.4f}\n")
    f.write(f"Baseline (Phase 2): {h2_baseline:.4f}  Delta: {h2_delta:+.4f}\n\n")
    f.write(h2_test_metrics["report_str"])
    f.write(f"\nConfusion Matrix ({H2_LABEL_NAMES}):\n")
    for row in h2_test_metrics["confusion_matrix"]:
        f.write("  " + "  ".join(f"{v:>5d}" for v in row) + "\n")

print(f"\nResults saved to {results_path}")

## Section 11: Training Curves

Six-panel visualization showing both heads' training progress side by side:
1. Training loss (total, Head 1, Head 2)
2. Val macro F1 for both heads
3. Head 1 per-class F1
4. Head 2 per-class F1
5. Head 1 test confusion matrix
6. Head 2 test confusion matrix

In [ ]:
epochs = history["epoch"]

fig = plt.figure(figsize=(20, 14))
fig.suptitle("Phase 4: Joint Two-Head Training Results",
             fontsize=16, fontweight="bold", y=0.98)
gs = gridspec.GridSpec(3, 2, hspace=0.5, wspace=0.35)

# Plot 1: Training Loss
ax1 = fig.add_subplot(gs[0, 0])
ax1.plot(epochs, history["train_loss_total"], "o-", color="#333333", linewidth=2, markersize=5, label="Total")
ax1.plot(epochs, history["train_loss_h1"], "s--", color="#4c8bf5", linewidth=1.5, markersize=4, label="Head 1")
ax1.plot(epochs, history["train_loss_h2"], "^--", color="#e05c5c", linewidth=1.5, markersize=4, label="Head 2")
ax1.set_title("Training Loss per Epoch")
ax1.set_xlabel("Epoch")
ax1.set_ylabel("Loss")
ax1.set_xticks(epochs)
ax1.legend(fontsize=8)
ax1.grid(True, alpha=0.3)

# Plot 2: Val Macro F1 both heads
ax2 = fig.add_subplot(gs[0, 1])
ax2.plot(epochs, history["h1_val_macro_f1"], "o-", color="#4c8bf5", linewidth=2, markersize=5, label="Head 1 F1")
ax2.plot(epochs, history["h2_val_macro_f1"], "s-", color="#e05c5c", linewidth=2, markersize=5, label="Head 2 F1")
ax2.axhline(y=0.8272, color="#4c8bf5", linestyle=":", linewidth=1, alpha=0.5, label="H1 Phase 1 baseline")
ax2.axhline(y=0.5091, color="#e05c5c", linestyle=":", linewidth=1, alpha=0.5, label="H2 Phase 2 baseline")
best_ep = epochs[history["h1_val_macro_f1"].index(max(history["h1_val_macro_f1"]))] if history["h1_val_macro_f1"] else 1
ax2.axvline(x=best_epoch, color="gray", linestyle=":", linewidth=1.5, label=f"Best epoch ({best_epoch})")
ax2.set_title("Val Macro F1: Both Heads")
ax2.set_xlabel("Epoch")
ax2.set_ylabel("Macro F1")
ax2.set_ylim(0, 1)
ax2.set_xticks(epochs)
ax2.legend(fontsize=7, loc="lower right")
ax2.grid(True, alpha=0.3)

# Plot 3: Head 1 per-class F1
ax3 = fig.add_subplot(gs[1, 0])
h1_colors = {
    "normal": "#34a853", "depression": "#4c8bf5",
    "anxiety": "#fbbc04", "stress": "#e05c5c", "suicidal": "#9b59b6"
}
for cls, color in h1_colors.items():
    ax3.plot(epochs, history[f"h1_val_{cls}_f1"], "o-", color=color,
             linewidth=2, markersize=5, label=cls.capitalize())
ax3.axhline(y=0.40, color="red", linestyle=":", linewidth=1, alpha=0.6, label="Stress warn (0.40)")
ax3.set_title("Head 1: Per-Class F1 over Epochs")
ax3.set_xlabel("Epoch")
ax3.set_ylabel("F1 Score")
ax3.set_ylim(0, 1)
ax3.set_xticks(epochs)
ax3.legend(fontsize=7, ncol=2)
ax3.grid(True, alpha=0.3)

# Plot 4: Head 2 per-class F1
ax4 = fig.add_subplot(gs[1, 1])
h2_colors = {
    "no_reason": "#34a853", "bias_or_abuse": "#e05c5c",
    "jobs_and_careers": "#fbbc04", "medication": "#9b59b6",
    "relationship": "#4c8bf5", "alienation": "#ff7f0e",
}
for cls, color in h2_colors.items():
    ax4.plot(epochs, history[f"h2_val_{cls}_f1"], "o-", color=color,
             linewidth=2, markersize=5, label=cls.replace("_", " ").title())
ax4.set_title("Head 2: Per-Class F1 over Epochs")
ax4.set_xlabel("Epoch")
ax4.set_ylabel("F1 Score")
ax4.set_ylim(0, 1)
ax4.set_xticks(epochs)
ax4.legend(fontsize=7, ncol=2)
ax4.grid(True, alpha=0.3)

# Plot 5: Head 1 test confusion matrix
ax5 = fig.add_subplot(gs[2, 0])
cm1 = h1_test_metrics["confusion_matrix"]
cm1_norm = cm1.astype(float) / cm1.sum(axis=1, keepdims=True)
im5 = ax5.imshow(cm1_norm, interpolation="nearest", cmap="Blues", vmin=0, vmax=1)
plt.colorbar(im5, ax=ax5, fraction=0.046, pad=0.04)
ax5.set_xticks(range(len(H1_LABEL_NAMES)))
ax5.set_yticks(range(len(H1_LABEL_NAMES)))
ax5.set_xticklabels(H1_LABEL_NAMES, rotation=35, ha="right", fontsize=8)
ax5.set_yticklabels(H1_LABEL_NAMES, fontsize=8)
ax5.set_title("Head 1: Test Confusion Matrix")
ax5.set_xlabel("Predicted")
ax5.set_ylabel("True")
for i in range(len(H1_LABEL_NAMES)):
    for j in range(len(H1_LABEL_NAMES)):
        val = cm1_norm[i, j]
        ax5.text(j, i, f"{val:.2f}", ha="center", va="center",
                 fontsize=7, color="white" if val > 0.6 else "black")

# Plot 6: Head 2 test confusion matrix
ax6 = fig.add_subplot(gs[2, 1])
cm2 = h2_test_metrics["confusion_matrix"]
cm2_norm = cm2.astype(float) / cm2.sum(axis=1, keepdims=True)
im6 = ax6.imshow(cm2_norm, interpolation="nearest", cmap="Oranges", vmin=0, vmax=1)
plt.colorbar(im6, ax=ax6, fraction=0.046, pad=0.04)
ax6.set_xticks(range(len(H2_LABEL_NAMES)))
ax6.set_yticks(range(len(H2_LABEL_NAMES)))
ax6.set_xticklabels(H2_LABEL_NAMES, rotation=35, ha="right", fontsize=7)
ax6.set_yticklabels(H2_LABEL_NAMES, fontsize=7)
ax6.set_title("Head 2: Test Confusion Matrix")
ax6.set_xlabel("Predicted")
ax6.set_ylabel("True")
for i in range(len(H2_LABEL_NAMES)):
    for j in range(len(H2_LABEL_NAMES)):
        val = cm2_norm[i, j]
        ax6.text(j, i, f"{val:.2f}", ha="center", va="center",
                 fontsize=6, color="white" if val > 0.6 else "black")

plt.savefig(os.path.join(CHECKPOINT_DIR, "joint_training_curves.png"), dpi=150, bbox_inches="tight")
plt.show()
print("Plot saved to checkpoints/joint_training_curves.png")

## Summary

All steps complete. Output files in `/kaggle/working/checkpoints/`:

| File | Contents |
|------|----------|
| `best_joint_checkpoint.pt` | Shared backbone + Head 1 + Head 2 weights |
| `joint_label_mapping.json` | Label mappings for both heads |
| `joint_test_results.txt` | Test evaluation for both heads with comparison to baselines |
| `joint_training_curves.png` | Six-panel training visualization |

Download these files from the Kaggle **Output** tab. The joint checkpoint is what gets used for Phase 5 (DIAC-WOZ adaptation).

In [ ]:
print("=" * 60)
print("OUTPUT FILES")
print("=" * 60)

for fname in sorted(os.listdir(CHECKPOINT_DIR)):
    fpath = os.path.join(CHECKPOINT_DIR, fname)
    size_mb = os.path.getsize(fpath) / (1024 * 1024)
    print(f"  {fname}  ({size_mb:.2f} MB)")

print(f"\nAll done. Download from the Output tab on Kaggle.")